# Pharma Sales Data — Cleaning & Exploration

This notebook loads the raw pharma sales dataset, checks it for quality issues (negative quantities, duplicate rows, zero-quantity rows), fixes what's fixable, engineers a proper `Date` column from `Month`/`Year`, and validates that `Quantity × Price = Sales`.

## 1. Setup

In [2]:
import pandas as pd

## 2. Load the Data

In [3]:
df= pd.read_csv(r"D:\pharma ai project\pharma-data.csv")

## 3. First Look at the Data

Basic shape, structure, and summary statistics before touching anything.

In [4]:
df.head()

,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team
0,Gottlieb-Cruickshank,"Zieme, Doyle and Kunze",Lublin,Poland,51.2333,22.5667,Hospital,Private,Topipizole,Mood Stabilizers,4.0,368,1472.0,January,2018,Mary Gerrard,Britanny Bold,Delta
1,Gottlieb-Cruickshank,Feest PLC,Świecie,Poland,53.4167,18.4333,Pharmacy,Retail,Choriotrisin,Antibiotics,7.0,591,4137.0,January,2018,Jessica Smith,Britanny Bold,Delta
2,Gottlieb-Cruickshank,Medhurst-Beer Pharmaceutical Limited,Rybnik,Poland,50.0833,18.5000,Pharmacy,Institution,Acantaine,Antibiotics,30.0,66,1980.0,January,2018,Steve Pepple,Tracy Banks,Bravo
3,Gottlieb-Cruickshank,Barton Ltd Pharma Plc,Czeladź,Poland,50.3333,19.0833,Hospital,Private,Lioletine Refliruvax,Analgesics,6.0,435,2610.0,January,2018,Mary Gerrard,Britanny Bold,Delta
4,Gottlieb-Cruickshank,Keeling LLC Pharmacy,Olsztyn,Poland,53.7800,20.4942,Pharmacy,Retail,Oxymotroban Fexoformin,Analgesics,20.0,458,9160.0,January,2018,Anne Wu,Britanny Bold,Delta


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 254082 entries, 0 to 254081
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Distributor        254082 non-null  str    
 1   Customer Name      254082 non-null  str    
 2   City               254082 non-null  str    
 3   Country            254082 non-null  str    
 4   Latitude           254082 non-null  float64
 5   Longitude          254082 non-null  float64
 6   Channel            254082 non-null  str    
 7   Sub-channel        254082 non-null  str    
 8   Product Name       254082 non-null  str    
 9   Product Class      254082 non-null  str    
 10  Quantity           254082 non-null  float64
 11  Price              254082 non-null  int64  
 12  Sales              254082 non-null  float64
 13  Month              254082 non-null  str    
 14  Year               254082 non-null  int64  
 15  Name of Sales Rep  254082 non-null  str    
 16  Manager      

In [6]:
df.shape

(254082, 18)

In [7]:
df.describe()

,Latitude,Longitude,Quantity,Price,Sales,Year
count,254082.000000,254082.000000,254082.000000,254082.000000,2.540820e+05,254082.000000
mean,50.962222,10.803212,112.872139,412.207720,4.643772e+04,2018.385187
std,1.625526,4.143311,744.310385,224.963687,3.491918e+05,1.041352
min,47.514200,6.083800,-7200.000000,22.000000,-4.161600e+06,2017.000000
25%,49.805600,7.891100,5.000000,195.000000,1.704000e+03,2018.000000
50%,51.133300,9.397800,20.000000,430.000000,5.850000e+03,2018.000000
75%,52.083300,12.133300,50.000000,605.000000,2.156525e+04,2019.000000
max,54.781900,23.566700,117600.000000,794.000000,7.420560e+07,2020.000000


In [8]:
df.describe(include="str")

,Distributor,Customer Name,City,Country,Channel,Sub-channel,Product Name,Product Class,Month,Name of Sales Rep,Manager,Sales Team
count,254082,254082,254082,254082,254082,254082,254082,254082,254082,254082,254082,254082
unique,29,751,749,2,2,4,240,6,12,13,4,4
top,Gerlach LLC,"Spinka, Schowalter and Conn",Friedberg,Germany,Pharmacy,Retail,Diprotaine,Antiseptics,November,Sheila Stones,Britanny Bold,Delta
freq,111364,440,796,213598,129971,68351,1143,52037,23323,19655,78244,78244


In [9]:
df.isnull().sum()

Distributor          0
Customer Name        0
City                 0
Country              0
Latitude             0
Longitude            0
Channel              0
Sub-channel          0
Product Name         0
Product Class        0
Quantity             0
Price                0
Sales                0
Month                0
Year                 0
Name of Sales Rep    0
Manager              0
Sales Team           0
dtype: int64

**Findings so far:**
- 254,082 rows, 18 columns, no missing values.
- `Quantity` ranges from **-7,200 to 117,600** — negative values need a closer look.
- `Sales` also goes negative (min ≈ **-4.16M**), consistent with negative quantities.

## 4. Data Quality Checks

### 4.1 Negative Quantities

Some rows have negative `Quantity` (and correspondingly negative `Sales`). Investigating how many and what they look like.

In [10]:
df[df["Quantity"]<0][["Product Name", "Product Class","Quantity", "Price", "Sales"]].head(10)

,Product Name,Product Class,Quantity,Price,Sales
3761,Exotropin Empizine,Mood Stabilizers,-19.0,785,-14915.0
3762,Epzipitant,Analgesics,-12.0,778,-9336.0
18526,Belavarix Benzabicin,Antibiotics,-1.0,131,-131.0
22613,Afinitasol,Antipiretics,-5.0,286,-1430.0
22751,Sucprine Specbalamin,Antiseptics,-50.0,182,-9100.0
22779,Amamadin,Antimalarial,-100.0,713,-71300.0
23355,Brimorix Amonine,Antibiotics,-2.0,398,-796.0
23631,Sevotrana Tiafoxin,Analgesics,-60.0,460,-27600.0
23855,Afluferon Entrarenone,Antimalarial,-10.0,104,-1040.0
23856,Androporin,Antiseptics,-10.0,659,-6590.0


In [11]:
df[df["Quantity"]<0].shape

(2633, 18)

In [12]:
df[df["Quantity"] < 0]["Sales"].lt(0).sum()

np.int64(2633)

In [13]:
df[df["Quantity"] < 0][
["Product Name", "Product Class", "Month", "Year",
     "Quantity", "Price", "Sales"]
].head(20)

,Product Name,Product Class,Month,Year,Quantity,Price,Sales
3761,Exotropin Empizine,Mood Stabilizers,February,2018,-19.0,785,-14915.0
3762,Epzipitant,Analgesics,February,2018,-12.0,778,-9336.0
18526,Belavarix Benzabicin,Antibiotics,November,2018,-1.0,131,-131.0
22613,Afinitasol,Antipiretics,February,2018,-5.0,286,-1430.0
22751,Sucprine Specbalamin,Antiseptics,March,2018,-50.0,182,-9100.0
22779,Amamadin,Antimalarial,March,2018,-100.0,713,-71300.0
23355,Brimorix Amonine,Antibiotics,May,2018,-2.0,398,-796.0
23631,Sevotrana Tiafoxin,Analgesics,June,2018,-60.0,460,-27600.0
23855,Afluferon Entrarenone,Antimalarial,July,2018,-10.0,104,-1040.0
23856,Androporin,Antiseptics,July,2018,-10.0,659,-6590.0


In [14]:
df[df["Quantity"] < 0]["Product Class"].value_counts()

Product Class
Antiseptics         532
Mood Stabilizers    498
Analgesics          432
Antipiretics        424
Antibiotics         381
Antimalarial        366
Name: count, dtype: int64

In [15]:
df[df["Quantity"] < 0]["Product Name"].nunique()

240

In [16]:
df["Product Name"].nunique()

240

**2,633 rows (~1%) have negative quantity, and every one of them also has negative sales** — the two move together consistently, so this doesn't look like a data-entry error, more likely returns/adjustments. They span all 240 product names and all 6 product classes, so it's not isolated to one product line. Flagged here; not dropped in this pass.

### 4.2 Duplicate Rows

Checking for exact duplicate rows and removing them.

In [17]:
df.duplicated().sum()

np.int64(4)

In [18]:
df[df.duplicated(keep=False)]

,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team
13821,Prohaska-Kuhic,"Steuber, Skiles and Kilback Pharma Plc",Sosnowiec,Poland,50.3000,19.1667,Pharmacy,Retail,Dasagestin,Analgesics,10.0,386,3860.0,September,2018,Mary Gerrard,Britanny Bold,Delta
24328,Prohaska-Kuhic,"Steuber, Skiles and Kilback Pharma Plc",Sosnowiec,Poland,50.3000,19.1667,Pharmacy,Retail,Dasagestin,Analgesics,10.0,386,3860.0,September,2018,Mary Gerrard,Britanny Bold,Delta
53977,Erdman,Fritsch LLC Pharmaceutical Ltd,Mülheim,Germany,51.4283,6.8789,Hospital,Government,Amphesirox,Antiseptics,30.0,25,750.0,April,2017,Jessica Smith,Britanny Bold,Delta
57178,Erdman,Fritsch LLC Pharmaceutical Ltd,Mülheim,Germany,51.4283,6.8789,Hospital,Government,Amphesirox,Antiseptics,30.0,25,750.0,April,2017,Jessica Smith,Britanny Bold,Delta
137363,Gerlach LLC,Crona PLC Pharmaceutical Ltd,Schloß Holte-Stukenbrock,Germany,51.8833,8.6167,Pharmacy,Institution,Thionide,Analgesics,10.0,113,1130.0,October,2018,Erica Jones,James Goodwill,Alfa
140251,Gerlach LLC,Crona PLC Pharmaceutical Ltd,Schloß Holte-Stukenbrock,Germany,51.8833,8.6167,Pharmacy,Institution,Thionide,Analgesics,10.0,113,1130.0,October,2018,Erica Jones,James Goodwill,Alfa
173838,Gerlach LLC,"Steuber, Skiles and Kilback Pharmacy",Datteln,Germany,51.6539,7.3417,Pharmacy,Institution,Interfeferon Aspipex,Antiseptics,10.0,416,4160.0,June,2019,Alan Ray,James Goodwill,Alfa
177689,Gerlach LLC,"Steuber, Skiles and Kilback Pharmacy",Datteln,Germany,51.6539,7.3417,Pharmacy,Institution,Interfeferon Aspipex,Antiseptics,10.0,416,4160.0,June,2019,Alan Ray,James Goodwill,Alfa


In [19]:
df = df.drop_duplicates()

In [20]:
df.duplicated().sum()

np.int64(0)

**4 exact duplicate rows found and dropped.**

### 4.3 Zero-Quantity Rows

Rows where `Quantity == 0` should also have `Sales == 0` — checking that, and whether these rows carry any useful information.

In [21]:
df[df["Quantity"] == 0].shape

(27, 18)

In [22]:
df[df["Quantity"] == 0][
    ["Product Name", "Product Class", "Quantity", "Price", "Sales", "Month", "Year"]
]

,Product Name,Product Class,Quantity,Price,Sales,Month,Year
4384,Novakyn Tracprox,Antiseptics,0.0,288,0.0,March,2018
4510,Tiaracil,Antipiretics,0.0,162,0.0,March,2018
4511,Sodinel Olofribrate,Antimalarial,0.0,50,0.0,March,2018
6554,Primapion,Antibiotics,0.0,492,0.0,April,2018
8643,Argalazine Abostryl,Antipiretics,0.0,628,0.0,June,2018
8667,Sevotrana Tiafoxin,Analgesics,0.0,460,0.0,June,2018
9055,Diprolimus,Antiseptics,0.0,634,0.0,June,2018
13779,Raparidol,Mood Stabilizers,0.0,187,0.0,September,2018
13791,Saprodeine Zanprosyn,Antibiotics,0.0,215,0.0,September,2018
13792,Formolovir Amanferon,Analgesics,0.0,191,0.0,September,2018


In [23]:
df[df["Quantity"]==0]["Sales"].value_counts()

Sales
0.0    27
Name: count, dtype: int64

In [24]:
df[df["Quantity"] == 0][
    ["Distributor", "Customer Name", "Product Name",
     "Quantity", "Price", "Sales", "Month", "Year"]
]

,Distributor,Customer Name,Product Name,Quantity,Price,Sales,Month,Year
4384,Prohaska-Kuhic,Daugherty-Rempel Pharma Plc,Novakyn Tracprox,0.0,288,0.0,March,2018
4510,Prohaska-Kuhic,Nader-Gaylord Pharmaceutical Limited,Tiaracil,0.0,162,0.0,March,2018
4511,Prohaska-Kuhic,Ritchie-Rodriguez Pharm,Sodinel Olofribrate,0.0,50,0.0,March,2018
6554,Prohaska-Kuhic,Zemlak Group Pharmaceutical Limited,Primapion,0.0,492,0.0,April,2018
8643,Stehr-Champlin,Kreiger Inc Pharma Plc,Argalazine Abostryl,0.0,628,0.0,June,2018
8667,Stehr-Champlin,"Parisian, Hagenes and Skiles Pharma Plc",Sevotrana Tiafoxin,0.0,460,0.0,June,2018
9055,Lindgren-Simonis Pharm,Hermiston and Sons Pharma Plc,Diprolimus,0.0,634,0.0,June,2018
13779,Rohan,Rogahn-Klein Pharmacy,Raparidol,0.0,187,0.0,September,2018
13791,Rohan,Koss-Beier,Saprodeine Zanprosyn,0.0,215,0.0,September,2018
13792,Rohan,"Wiegand, Jast and Yost Pharma Plc",Formolovir Amanferon,0.0,191,0.0,September,2018


In [25]:
df=df[df["Quantity"]!=0].copy()

In [26]:
df["Quantity"].eq(0).sum()

np.int64(0)

**27 rows had `Quantity == 0` and `Sales == 0`** — no transaction actually happened, so these were removed.

In [27]:
df.shape

(254051, 18)

Dataset is now **254,051 rows** after removing duplicates and zero-quantity rows.

## 5. Feature Engineering: Date Column

`Month` (name) and `Year` (int) are separate columns — combining them into a single proper `datetime` column.

In [28]:
df["Month"].unique()

<StringArray>
[  'January',  'February',     'March',     'April',       'May',      'June',
      'July',    'August', 'September',   'October',  'November',  'December']
Length: 12, dtype: str

In [29]:
df["Date"]= pd.to_datetime(
    df["Month"] + " " + df["Year"].astype(str),
    format= "%B %Y"
)

In [30]:
df[["Month", "Year" ,"Date"]].head()

,Month,Year,Date
0,January,2018,2018-01-01
1,January,2018,2018-01-01
2,January,2018,2018-01-01
3,January,2018,2018-01-01
4,January,2018,2018-01-01


In [31]:
df["Date"].dtype

dtype('<M8[us]')

## 6. Cleaning Summary

In [32]:
print("Rows:", len(df))
print("Duplicates:", df.duplicated().sum())
print("Zero quantity:", (df["Quantity"] == 0).sum())

Rows: 254051
Duplicates: 0
Zero quantity: 0


## 7. Sales Formula Check

Sanity check: does `Quantity × Price` always equal `Sales`? If not, how many rows disagree and by how much?

In [33]:
(df["Quantity"] * df["Price"] != df["Sales"]).sum()

np.int64(29)

In [34]:
mismatch = df["Quantity"] * df["Price"] != df["Sales"]
df.loc[mismatch, [
    "Quantity", "Price", "Sales",
    "Product Name", "Product Class", "Month", "Year"
]]

,Quantity,Price,Sales,Product Name,Product Class,Month,Year
5706,0.200000,34,6.800000,Proderal,Antiseptics,April,2018
71153,163.551276,162,26495.306690,Tiaracil,Antipiretics,July,2017
71154,163.551276,634,103691.508900,Diprolimus,Antiseptics,July,2017
169774,175.879397,319,56105.527640,Amcibax Amcikeran,Analgesics,May,2019
169775,140.703518,201,28281.407040,Palodizem,Mood Stabilizers,May,2019
169782,189.698492,409,77586.683420,Halciramine,Antimalarial,May,2019
169783,131.909548,590,77826.633170,Decidenu Galantrigine,Antipiretics,May,2019
169784,151.758794,41,6222.110553,Interfedox Pilobamol,Antiseptics,May,2019
169785,105.527638,319,33663.316580,Amcibax Amcikeran,Analgesics,May,2019
169786,2.512563,460,1155.778894,Sevotrana Tiafoxin,Analgesics,May,2019


**29 rows don't satisfy `Quantity × Price = Sales` exactly.** Looking at the mismatches, `Quantity` in these rows has non-integer, oddly precise values (e.g. `163.551276`, `1164.536741`) instead of the clean whole numbers seen elsewhere — these look like they were back-calculated from `Sales / Price` rather than being original input, so `Sales` is likely the reliable field here rather than `Quantity`. Worth deciding how to handle these before using `Quantity` downstream.

## 8. Final Cleanup & Export

Text columns can carry stray leading/trailing whitespace from the source file — stripping those, then exporting the cleaned dataset to CSV for downstream use.

In [36]:
text_cols = df.select_dtypes(include="str").columns

for col in text_cols:
    df[col] = df[col].str.strip()

In [37]:
df.to_csv("pharma_sales_clean.csv", index=False)

**`pharma_sales_clean.csv` now holds the cleaned dataset** — deduped, zero-quantity rows removed, whitespace-trimmed text fields, and a proper `Date` column. The negative-quantity rows (2,633) and the 29 `Quantity × Price ≠ Sales` mismatches are still in there, unresolved — same open item as before.